In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import librosa
import os
import json
from tqdm import tqdm
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

In [2]:
class ImprovedChromaDataset(Dataset):
    def __init__(self, audio_dir, label_dir, num_classes=4, sr=16000, window_size=20, hop_length=512, target_bins=25, file_list=None):
        self.audio_dir = audio_dir
        if file_list is not None: self.audio_files = file_list
        else: self.audio_files = [f for f in os.listdir(audio_dir) if f.endswith(('.wav', '.mp3'))]
        self.audio_segments = self.get_segments()

        self.num_classes = num_classes
        self.label_dir = label_dir
        with open(label_dir, "r", encoding="utf-8") as file: self.label_data = json.load(file)
        self.mode_mapping = {"Unknown": 0, "우조": 1, "계면조": 2, "아니리": 3}

        self.sr = sr
        self.target_bins = target_bins
        self.window_frames = window_size * sr // hop_length

        self.chroma_cache, self.label_cache = {}, {}

    def get_audio_duration(self, file_path):
        info = librosa.get_duration(filename=file_path)
        return info
            
    def get_segments(self):
        segments = []
        for file_name in self.audio_files:
            file_path = os.path.join(self.audio_dir, file_name)
            duration = self.get_audio_duration(file_path)

            num_segments = max(3, int(duration // 10))  # 세그먼트 길이 단축 (20초 → 10초)
            step = (duration - 10) / num_segments
            start_times = [max(0, min(duration - 10, i * step + np.random.uniform(-1, 1))) for i in range(num_segments)]
                          
            file_segments = [(file_name, start) for start in start_times]
            segments.extend(file_segments)            
        return segments

    def extract_chroma(self, file_path, start_time=0, duration=20):
        cache_key = f"{file_path}_{start_time}_{duration}" # 캐싱 키 생성
        if cache_key in self.chroma_cache: return self.chroma_cache[cache_key] # 캐시에 있으면 반환

        y, sr = librosa.load(file_path, sr=self.sr, offset=start_time, duration=duration) # 지정된 구간만 로드
        chroma = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=512) # 다양한 특성 추출 (크로마 + MFCC)
        chroma = self.expand_chroma(chroma) # 타겟 빈 수에 맞게 크로마 특성 확장
        self.chroma_cache[cache_key] = chroma # 결과 캐싱        
        return chroma

    def expand_chroma(self, chroma):
        repeat = self.target_bins // chroma.shape[0] + 1
        expanded_chroma = np.tile(chroma, (repeat, 1))
        return expanded_chroma[:self.target_bins, :]

    def get_label(self, file_name, num_frames, start_time=0, duration=20):
        cache_key = f"{file_name}_{num_frames}_{start_time}_{duration}" # 캐싱 키 생성
        if cache_key in self.label_cache: return self.label_cache[cache_key] # 캐시에 있으면 반환
            
        hash_value = file_name.split("-")[0]
        labels = np.zeros((num_frames, self.num_classes))
        for item in self.label_data:
            if item.get("file_upload", "").startswith(hash_value):                
                for annotation in item.get("annotations", []):
                    for result in annotation.get("result", []):
                        value = result.get("value", {})
                        original_start = value.get("start", 0)
                        original_end = value.get("end", 0)
                        
                        rel_start = max(0, original_start - start_time)
                        rel_end = min(duration, original_end - start_time)
                        
                        if rel_end > 0 and rel_start < duration:
                            mode_label = self.mode_mapping.get(value.get("labels", ["Unknwon"])[0], 0)                            
                            start_frame = int((rel_start / duration) * num_frames)
                            end_frame = int((rel_end / duration) * num_frames)

                            if start_frame < num_frames and end_frame > 0:
                                start_frame = max(0, start_frame)
                                end_frame = min(num_frames, end_frame)
                                labels[start_frame:end_frame, mode_label] = 1

        self.label_cache[cache_key] = labels
        return labels

    def __len__(self):
        return len(self.audio_segments)

    def __getitem__(self, idx):
        duration = 20  # fixed size

        file_name, start_time = self.audio_segments[idx]
        file_path = os.path.join(self.audio_dir, file_name)
        
        chroma = self.extract_chroma(file_path, start_time, duration)
        num_frames = chroma.shape[1]
        
        labels = self.get_label(file_name, num_frames, start_time, duration)
        

        if num_frames < self.window_frames:
            chroma_pad = np.zeros((self.target_bins, self.window_frames - num_frames))
            chroma = np.concatenate((chroma, chroma_pad), axis=1)
            label_pad = np.zeros((self.window_frames - num_frames, self.num_classes))
            labels = np.concatenate((labels, label_pad), axis=0)
        elif num_frames > self.window_frames:
            max_start = num_frames - self.window_frames
            start_idx = np.random.randint(0, max_start + 1) if max_start > 0 else 0
            chroma = chroma[:, start_idx:start_idx + self.window_frames]
            labels = labels[start_idx:start_idx + self.window_frames]
        

        chroma_tensor = torch.tensor(chroma, dtype=torch.float32).unsqueeze(0)
        
        if labels.shape[1] == self.num_classes: label_tensor = torch.tensor(np.argmax(label, axis=1), dtype=torch.long).unsqueeze(1)
        else: label_tensor = torch.tensor(labels, dtype=torch.long).unsqueeze(1)
        
        return chroma_tensor, label_tensor

In [3]:
class CNNChromagram(nn.Module):
    def __init__(self, target_bin=25, num_classes=4):
        super(CNNChromagram, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3, 3), padding='same', dilation=(1, 2))
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 3), padding='same', dilation=(1, 2))
        self.conv3 = nn.Conv2d(64, 128, kernel_size=(3, 3), padding='same', dilation=(1, 2))

        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=(2, 1))

        fc_input_size = 128 * (target_bin // 8)  # 3번의 풀링 (2^3 = 8로 나눔)

        self.fc1 = nn.Linear(fc_input_size, 64)
        self.fc2 = nn.Linear(64, num_classes)
        self.dropout = nn.Dropout(0.5)  # 드롭아웃 증가로 과적합 방지

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)

        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)

        x = self.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)

        batch_size, channels, freq_bins, time_steps = x.shape

        x = x.permute(0, 3, 1, 2)
        x = x.reshape(batch_size, time_steps, -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [4]:
def train_and_evaluate(model, trainloader, testloader, criterion, optimizer, device, epochs=50):
    train_losses, train_accuracies = [], []
    val_losses, val_accuracies = [], []
    
    for epoch in tqdm(range(epochs), desc="Training"):
        model.train()
        running_loss = 0.0
        correct_train, total_train = 0, 0
        
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)  # [batch, time, classes]
            
            batch_size, seq_len, num_classes = outputs.shape
            outputs_reshaped = outputs.permute(0, 2, 1)  # [batch, classes, time]
            targets_reshaped = targets.squeeze(2).long()  # [batch, time]
            
            loss = criterion(outputs_reshaped, targets_reshaped)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            _, predicted = torch.max(outputs, 2)
            total_train += targets_reshaped.numel()
            correct_train += (predicted == targets_reshaped).sum().item()
        
        avg_train_loss = running_loss / len(trainloader)
        train_accuracy = 100.0 * correct_train / total_train
        
        train_losses.append(avg_train_loss)
        train_accuracies.append(train_accuracy)
        

        model.eval()
        val_loss = 0.0
        correct_val, total_val = 0, 0        
        with torch.no_grad():
            for inputs, targets in testloader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                
                batch_size, seq_len, num_classes = outputs.shape
                outputs_reshaped = outputs.permute(0, 2, 1)  # [batch, classes, time]
                targets_reshaped = targets.squeeze(2).long()  # [batch, time]
                
                loss = criterion(outputs_reshaped, targets_reshaped)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs, 2)
                total_val += targets_reshaped.numel()
                correct_val += (predicted == targets_reshaped).sum().item()
        
        avg_val_loss = val_loss / len(testloader)
        val_accuracy = 100.0 * correct_val / total_val
        
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)
        
        if (epoch + 1) % 5 == 0:
            print(f'Epoch {epoch+1}/{epochs}: '
                  f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, '
                  f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')
    
    return {'train_losses': train_losses, 'train_accuracies': train_accuracies,
            'val_losses': val_losses, 'val_accuracies': val_accuracies}

In [5]:
def analyze_results(all_results, n_splits):
    final_train_accs = [results['train_accuracies'][-1] for results in all_results]
    final_val_accs = [results['val_accuracies'][-1] for results in all_results]
    
    print("\n\n=== K-Fold Cross Validation Results ===")
    print(f"Average Training Accuracy: {np.mean(final_train_accs):.2f}% ± {np.std(final_train_accs):.2f}%")
    print(f"Average Validation Accuracy: {np.mean(final_val_accs):.2f}% ± {np.std(final_val_accs):.2f}%")
    
    plt.figure(figsize=(15, 10))
    
    plt.subplot(2, 1, 1)
    for fold in range(n_splits):
        plt.plot(all_results[fold]['train_losses'], label=f'Fold {fold+1} Train')
        plt.plot(all_results[fold]['val_losses'], label=f'Fold {fold+1} Val', linestyle='--')
    plt.title('Loss Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(2, 1, 2)
    for fold in range(n_splits):
        plt.plot(all_results[fold]['train_accuracies'], label=f'Fold {fold+1} Train')
        plt.plot(all_results[fold]['val_accuracies'], label=f'Fold {fold+1} Val', linestyle='--')
    plt.title('Accuracy Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig('kfold_learning_curves.png')
    plt.show()

In [6]:
def run_kfold_training():
    audio_dir = '/Users/jcastle/workspace/pansori/-/Pansori_2025_ISMIR/PansoriData/Audio/new'
    label_dir = '/Users/jcastle/workspace/pansori/-/Pansori_2025_ISMIR/PansoriData/label.json'
    
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")
    
    all_files = [f for f in os.listdir(audio_dir) if f.endswith(('.wav', '.mp3'))]
    
    n_splits = 5
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    all_results = []
    
    for fold, (train_indices, test_indices) in enumerate(kfold.split(all_files)):
        print(f"\n{'='*50}")
        print(f"FOLD {fold+1}/{n_splits}")
        print(f"{'='*50}")
        
        train_files = [all_files[i] for i in train_indices]
        test_files = [all_files[i] for i in test_indices]
        
        print(f"Training on {len(train_files)} files, Testing on {len(test_files)} files")
        
        train_dataset = ImprovedChromaDataset(audio_dir, label_dir, file_list=train_files)
        test_dataset = ImprovedChromaDataset(audio_dir, label_dir, file_list=test_files)

        trainloader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
        testloader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)
        # trainloader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)
        # testloader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4)
        

        model = CNNChromagram(target_bin=25, num_classes=4).to(device)
        optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
        criterion = nn.CrossEntropyLoss()

        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)
        
        results = train_and_evaluate(model, trainloader, testloader, criterion, optimizer, device, epochs=50)
        
        all_results.append(results)
        torch.save(model.state_dict(), f'model_fold_{fold+1}.pth')
        scheduler.step(results['val_losses'][-1])

    # analyze_results(all_results, n_splits)

In [7]:
run_kfold_training()

Using device: mps

FOLD 1/5
Training on 7 files, Testing on 2 files


/var/folders/nl/c_3x6g0n68x745l9b_dd0mmh0000gn/T/ipykernel_33998/2671947243.py:20: FutureWarning: get_duration() keyword argument 'filename' has been renamed to 'path' in version 0.10.0.
	This alias will be removed in version 1.0.
  info = librosa.get_duration(filename=file_path)
/Users/jcastle/workspace/pansori/pansori_venv/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Training:   0%|          | 0/50 [00:00<?, ?it/s]


NameError: name 'label' is not defined